# Adversarial Testing of the Safety Layer
### Stage 5: Trying, on Purpose, to Break `src/safety_layer.py`

`01_agent_overview.ipynb` explained the safety layer's design. This notebook does the opposite of trusting that design — it throws deliberately malformed, hostile, and edge-case LLM outputs at it and checks two invariants that must hold no matter what:

1. `validate_schema()` / `validate_constraints()` / `run_safety_checks()` **never raise** — they always return a `ValidationResult`, even on garbage input.
2. `plan_with_retry()` **never crashes and never acts on an unverified plan**, even when every single retry attempt is also garbage.

This is not a hypothetical exercise — adversarial testing found a real bug in this project (Section 3 below), which was fixed as a direct result.


In [ ]:
import sys
sys.path.insert(0, "../src")

import json
from schemas import NetworkRules, SliceState
from safety_layer import validate_schema, run_safety_checks
from agent_planner import plan_with_retry

RULES = NetworkRules(total_capacity_mbps=100.0, urllc_min_guarantee_mbps=30.0, max_step_change_mbps=20.0)
PREV_ALLOC = {"URLLC": 40.0, "eMBB": 60.0}
SLICE_STATES = [
    SliceState("URLLC", 40.0, 5.0, 38.0, 42.0, 4.0),
    SliceState("eMBB", 60.0, 12.0, 58.0, 55.0, 8.0),
]
print("setup ready")


## 1. Schema-Level Attacks

Malformed JSON structure, wrong types, missing fields, empty strings — none of these should ever reach an `AllocationPlan` object. `validate_schema()` should catch every one of them, with a specific, actionable error message (the exact string that gets fed back into the retry prompt — `notebooks/01_agent_overview.ipynb`, Section 6).


In [ ]:
schema_attacks = [
    {},                                                                        # empty object
    {"timestep": 1},                                                          # missing everything else
    {"timestep": "one", "allocations": {"URLLC": 40}, "reasoning": "x"},        # wrong type for timestep
    {"timestep": 1, "allocations": "give it all to URLLC", "reasoning": "x"},   # allocations not a dict
    {"timestep": 1, "allocations": {}, "reasoning": "x"},                       # empty allocations
    {"timestep": 1, "allocations": {"URLLC": "lots", "eMBB": 10}, "reasoning": "x"},  # non-numeric value
    {"timestep": 1, "allocations": {"URLLC": 40.0, "eMBB": 60.0}},              # missing reasoning
    {"timestep": 1, "allocations": {"URLLC": 40.0, "eMBB": 60.0}, "reasoning": ""},   # empty reasoning
    [1, 2, 3],                                                                  # not an object at all
    "just a plain string, not even JSON-shaped",
]

for attack in schema_attacks:
    result = validate_schema(attack)
    status = "REJECTED" if not result.ok else "!! ACCEPTED !!"
    print(f"{status:16s} {str(attack)[:55]:55s} -> {result.error}")


## 2. Constraint-Level Attacks

These are all **schema-valid** — well-formed JSON, correct types — but physically or operationally nonsensical. This is exactly the class of failure the safety layer's *second* stage exists for (`notebooks/01_agent_overview.ipynb`, Section 5B): a plan can be perfectly well-formed and still be a plan that would break the network.


In [ ]:
constraint_attacks = [
    {"timestep": 1, "allocations": {"URLLC": -50.0, "eMBB": 60.0}, "reasoning": "x"},   # negative
    {"timestep": 1, "allocations": {"URLLC": 200.0, "eMBB": 200.0}, "reasoning": "x"},  # way over capacity
    {"timestep": 1, "allocations": {"URLLC": 5.0, "eMBB": 95.0}, "reasoning": "x"},     # below URLLC minimum
    {"timestep": 1, "allocations": {"eMBB": 60.0}, "reasoning": "x"},                   # URLLC omitted
    {"timestep": 1, "allocations": {"URLLC": 90.0, "eMBB": 10.0}, "reasoning": "x"},    # step change too big
]

for attack in constraint_attacks:
    result = run_safety_checks(attack, RULES, PREV_ALLOC)
    status = "REJECTED" if not result.ok else "!! ACCEPTED !!"
    print(f"{status:16s} [{result.stage:11s}] {attack['allocations']} -> {result.error}")


## 3. A Real Bug This Testing Found: NaN and Infinity

Adversarial testing isn't just a formality — it caught an actual gap in this project. `float('nan')` is a legal Python/JSON-adjacent numeric value, and **NaN comparisons are always `False`**: `nan < 0` is `False`, `nan > capacity` is `False`. Before the fix below, a plan containing `NaN` for an allocation **passed both safety checks silently**, because every constraint comparison against it silently evaluated to "no violation found."

The fix: `validate_schema()` now explicitly checks `math.isfinite()` on every allocation value, rejecting `NaN` and `inf` at the schema stage before they can reach any numeric comparison at all. (While fixing this, the same check was extended to reject Python `bool` values too — `True`/`False` pass `isinstance(x, (int, float))` in Python since `bool` is a subclass of `int`, which would otherwise let `{"URLLC": true}` silently mean `{"URLLC": 1.0}`.)


In [ ]:
nan_inf_attacks = [
    {"timestep": 1, "allocations": {"URLLC": float("nan"), "eMBB": 60.0}, "reasoning": "x"},
    {"timestep": 1, "allocations": {"URLLC": float("inf"), "eMBB": 60.0}, "reasoning": "x"},
    {"timestep": 1, "allocations": {"URLLC": True, "eMBB": 60.0}, "reasoning": "x"},
]

for attack in nan_inf_attacks:
    result = validate_schema(attack)
    status = "REJECTED" if not result.ok else "!! ACCEPTED (bug!) !!"
    print(f"{status:24s} {attack['allocations']} -> {result.error}")


## 4. Worst Case — the LLM Returns Garbage on Every Single Retry

The previous sections tested individual functions directly. This tests the **full `plan_with_retry()` loop** under the worst realistic scenario: an LLM (or reference stand-in) that returns a different piece of garbage on the initial attempt *and* every retry, chosen at random from every attack type above. The bounded retry loop must exhaust its attempts and fall back — never crash, never act on any of it.


In [ ]:
import random

ALL_ATTACKS = schema_attacks + constraint_attacks + nan_inf_attacks
rng = random.Random(7)

def chaotic_llm(system_prompt, user_prompt):
    choice = rng.choice(ALL_ATTACKS)
    return json.dumps(choice) if isinstance(choice, (dict, list)) else str(choice)

plan = plan_with_retry(41, SLICE_STATES, RULES, PREV_ALLOC, chaotic_llm)

print("Final plan after every attempt was adversarial garbage:")
print(json.dumps(plan.to_dict(), indent=2))

assert "Fallback policy" in plan.reasoning
assert plan.allocations["URLLC"] >= RULES.urllc_min_guarantee_mbps
assert sum(plan.allocations.values()) <= RULES.total_capacity_mbps
print("\nNo crash. No unsafe plan reached actuation. Fallback triggered correctly, every time.")


## 5. What This Does and Doesn't Prove

- **Proves:** the safety layer is a genuine, tested backstop — not just documentation of intent. Every attack above is a real, executed test, not a hypothetical.
- **Does not prove:** that the LLM's *good-faith, schema-valid, constraint-respecting* plans are good decisions. A plan can be perfectly safe and still be a poor allocation choice — the safety layer's job is narrowly "never let through something that breaks the network," not "guarantee optimal performance." That gap is exactly what the debugging story in `notebooks/04_agentic_planner.ipynb`, Section 2, is about.


## 6. What's Implemented Where

| Attack class | Test file | Count |
|---|---|---|
| Schema-level (malformed structure/types) | `tests/test_safety_layer_adversarial.py` | 14 cases, 1 test |
| Constraint-level (valid JSON, unsafe physically) | `tests/test_safety_layer_adversarial.py` | 5 cases, 1 test |
| Full chaotic retry-loop worst case | `tests/test_safety_layer_adversarial.py` | 1 test |
| LLM callable itself raising (network error) | `tests/test_safety_layer_adversarial.py` | 1 test (documents current propagate-don't-swallow behaviour) |
| Original core safety layer tests (Stage 1) | `tests/test_safety_layer.py` | 9 tests |

All 14 safety-layer tests (5 adversarial + 9 original) plus the rest of the suite — 43 tests total across the whole repo — pass. Run with `PYTHONPATH=src pytest tests/ -v`.

---
*Next: → `06_baseline_comparison.ipynb`*
